In [2]:
import pandas as pd
import numpy as np

# Load raw datasets
df_train = pd.read_csv("train.csv", parse_dates=['Date'], low_memory=False)
df_store = pd.read_csv("store.csv", low_memory=False)

# 1. Dimensions & Data Memory Footprint
print("=== TRAIN DATASET DIMENSIONS ===")
print(f"Total Sales Records: {len(df_train):,}")
print(f"Total Columns: {df_train.shape[1]}")

print("\n=== STORE METADATA DIMENSIONS ===")
print(f"Total Stores: {len(df_store):,}")
print(f"Total Columns: {df_store.shape[1]}")

=== TRAIN DATASET DIMENSIONS ===
Total Sales Records: 1,017,209
Total Columns: 9

=== STORE METADATA DIMENSIONS ===
Total Stores: 1,115
Total Columns: 10


In [3]:
# 2. Schema Structure & Null Matrix
print("\n=== TRAIN DATASET INFO ===")
print(df_train.info())

print("\n=== STORE METADATA MISSING VALUES ===")
print(df_store.isnull().sum())


=== TRAIN DATASET INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[us]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  str           
 8   SchoolHoliday  1017209 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 70.8 MB
None

=== STORE METADATA MISSING VALUES ===
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOp

In [4]:
# 3. Target Variable Inspection
print("\n=== SALES SUMMARY METRICS ===")
print(df_train['Sales'].describe())


=== SALES SUMMARY METRICS ===
count    1.017209e+06
mean     5.773819e+03
std      3.849926e+03
min      0.000000e+00
25%      3.727000e+03
50%      5.744000e+03
75%      7.856000e+03
max      4.155100e+04
Name: Sales, dtype: float64


In [5]:
# 4. Open vs Closed Days Distribution
print("\n=== STORE OPEN STATUS DISTRIBUTION ===")
print(df_train['Open'].value_counts(normalize=True) * 100)


=== STORE OPEN STATUS DISTRIBUTION ===
Open
1    83.010669
0    16.989331
Name: proportion, dtype: float64


In [8]:
# Filter open store days (eliminates 0 sales skewness)
df_train = df_train[df_train['Open'] == 1].copy()
    
# Impute missing values
df_store['CompetitionDistance'] = df_store['CompetitionDistance'].fillna(df_store['CompetitionDistance'].median())
df_store['Promo2SinceWeek'] = df_store['Promo2SinceWeek'].fillna(0)
    
# Relational Join on 'Store' column
df_merged = pd.merge(df_train, df_store, on='Store', how='left')
    
# Crucial: Sort chronologically before lag calculation
df_merged = df_merged.sort_values(by=['Store', 'Date']).reset_index(drop=True)
    
df_merged.to_csv("rossmann_cleaned.csv", index=False)
print(f"[+] Cleaned data saved: {len(df_merged):,} records.")



[+] Cleaned data saved: 844,392 records.


In [9]:
import pandas as pd
import numpy as np

def run_data_pipeline(train_path: str, store_path: str) -> pd.DataFrame:
    print("[+] Initiating Data Ingestion Pipeline...")
    
    # 1. Ingest Raw Datasets
    df_train = pd.read_csv(train_path, parse_dates=['Date'], low_memory=False)
    df_store = pd.read_csv(store_path, low_memory=False)
    
    print(f"[+] Raw Records Loaded - Train: {len(df_train):,}, Store Metadata: {len(df_store):,}")
    
    # 2. Filter Closed Days (Open == 1) to eliminate zero-demand standard deviation skews
    df_train = df_train[df_train['Open'] == 1].copy()
    print(f"[+] Records Filtered (Open Days Only): {len(df_train):,}")
    
    # 3. Handle Missing Values in Store Metadata
    df_store['CompetitionDistance'] = df_store['CompetitionDistance'].fillna(df_store['CompetitionDistance'].median())
    df_store['CompetitionOpenSinceMonth'] = df_store['CompetitionOpenSinceMonth'].fillna(0)
    df_store['CompetitionOpenSinceYear'] = df_store['CompetitionOpenSinceYear'].fillna(0)
    df_store['Promo2SinceWeek'] = df_store['Promo2SinceWeek'].fillna(0)
    df_store['Promo2SinceYear'] = df_store['Promo2SinceYear'].fillna(0)
    df_store['PromoInterval'] = df_store['PromoInterval'].fillna('None')
    
    # 4. Merge Store Metadata with Historical Sales
    df_merged = pd.merge(df_train, df_store, on='Store', how='left')
    
    # 5. Temporal Filtering & Chronological Sorting
    df_merged = df_merged.sort_values(by=['Store', 'Date']).reset_index(drop=True)
    
    # 6. Save Processed Analytical Dataset
    df_merged.to_csv("rossmann_cleaned.csv", index=False)
    print(f"[+] Cleaned Merged Schema saved to 'rossmann_cleaned.csv'. Total Rows: {len(df_merged):,}")
    return df_merged

# Run Data Ingestion Pipeline
df_cleaned = run_data_pipeline("train.csv", "store.csv")

[+] Initiating Data Ingestion Pipeline...
[+] Raw Records Loaded - Train: 1,017,209, Store Metadata: 1,115
[+] Records Filtered (Open Days Only): 844,392
[+] Cleaned Merged Schema saved to 'rossmann_cleaned.csv'. Total Rows: 844,392
